# 04. Таблица `user_activity_types.csv`

Ноутбук присваивает пользователям основной тип слушателя и дополнительный временной профиль на основе таблицы `data/processed/user_features.csv`.

## 1. Импорт библиотек

In [9]:
from pathlib import Path

import pandas as pd

## 2. Пути проекта и выходной файл

In [10]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR            = PROJECT_ROOT / "data" / "processed"
USER_FEATURES_PATH       = PROCESSED_DIR / "user_features.csv"
USER_ACTIVITY_TYPES_PATH = PROCESSED_DIR / "user_activity_types.csv"

USER_FEATURES_PATH, USER_ACTIVITY_TYPES_PATH

(PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_features.csv'),
 PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_activity_types.csv'))

## 3. Константы и правила классификации

In [11]:
FLOAT_COLUMNS = [
    "active_days_share",
    "avg_daily_plays",
    "max_daily_plays",
    "evening_share",
    "night_share",
    "avg_session_length",
    "weekend_share",
]

REQUIRED_COLUMNS = [
    "username",
    "total_plays",
    "active_days",
    "morning_share",
    "day_share",
    "main_day_part",
    "peak_hour",
    "session_count",
] + FLOAT_COLUMNS

TYPE_DESCRIPTIONS = {
    "Интенсивный":
        "Пользователь с высоким средним числом прослушиваний в активный день.",
    "Регулярный":
        "Пользователь со стабильной активностью без экстремально высокой интенсивности.",
    "Эпизодический":
        "Пользователь с низкой долей активных дней и нерегулярным прослушиванием.",
}

DAY_PART_PROFILE_MAP = {
    "Ночь":  "Ночная активность",
    "Утро":  "Утренняя активность",
    "День":  "Дневная активность",
    "Вечер": "Вечерняя активность",
}

MIXED_ACTIVITY_GAP_THRESHOLD = 0.10

## 4. Функции присвоения типов слушателей и временных профилей

In [12]:
def validate_columns(df: pd.DataFrame):
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    assert len(missing_columns) == 0, (
        f"Missing required columns in user_features_df: {missing_columns}"
    )


def get_threshold(series: pd.Series, quantile: float) -> float:
    return float(series.quantile(quantile))


def build_behavior_profile(row: pd.Series) -> tuple[str, str]:
    shares = [
        (share, float(row[share])) for share in (
                'morning_share',
                'day_share',
                'evening_share',
                'night_share'
            )
    ]
    shares.sort(key=lambda item: item[1], reverse=True)

    top_share = {
        'name': (shares[0])[0],
        'value': (shares[0])[1],
    }
    second_share = {
        'name': (shares[1])[0],
        'value': (shares[1])[1],
    }

    share_gap = top_share['value'] - second_share['value']

    if share_gap < MIXED_ACTIVITY_GAP_THRESHOLD:
        return (
            "Смешанная активность",
            (
                f"share_gap < {MIXED_ACTIVITY_GAP_THRESHOLD:.2f}; "
                f"top_share={top_share['name']}; second_share={second_share['name']}"
            ),
        )

    main_day_part = DAY_PART_PROFILE_MAP.get(row["main_day_part"])
    assert main_day_part is not None, (
        f"Unsupported main_day_part value: {row['main_day_part']}"
    )

    return (
        main_day_part,
        (
            f"share_gap >= {MIXED_ACTIVITY_GAP_THRESHOLD:.2f}; "
            f"main_day_part={row['main_day_part']}"
        ),
    )


def assign_listener_types(user_features_df: pd.DataFrame) -> pd.DataFrame:
    validate_columns(user_features_df)

    user_features_df = user_features_df.copy().sort_values("username", kind="stable").reset_index(drop=True)

    high_intensity_threshold = get_threshold(user_features_df["avg_daily_plays"], 0.75)
    low_activity_threshold   = get_threshold(user_features_df["active_days_share"], 0.25)

    rows = []

    for _, row in user_features_df.iterrows():
        if row["avg_daily_plays"] >= high_intensity_threshold:
            listener_type   = "Интенсивный"
            assignment_rule = "avg_daily_plays >= upper_quartile"
        elif row["active_days_share"] <= low_activity_threshold:
            listener_type   = "Эпизодический"
            assignment_rule = "active_days_share <= lower_quartile"
        else:
            listener_type   = "Регулярный"
            assignment_rule = "default regular listener rule"

        behavior_profile, behavior_rule = build_behavior_profile(row)

        rows.append(
            {
                "username":                  row["username"],
                "listener_type":             listener_type,
                "listener_type_description": TYPE_DESCRIPTIONS[listener_type],
                "behavior_profile":          behavior_profile,
                "assignment_rule":           assignment_rule,
                "behavior_rule":             behavior_rule,
                "total_plays":               int(row["total_plays"]),
                "active_days":               int(row["active_days"]),
                "active_days_share":         row["active_days_share"],
                "avg_daily_plays":           row["avg_daily_plays"],
                "max_daily_plays":           int(row["max_daily_plays"]),
                "evening_share":             row["evening_share"],
                "night_share":               row["night_share"],
                "weekend_share":             row["weekend_share"],
                "main_day_part":             row["main_day_part"],
                "peak_hour":                 int(row["peak_hour"]),
                "session_count":             int(row["session_count"]),
                "avg_session_length":        row["avg_session_length"],
            }
        )

    result = pd.DataFrame(rows).sort_values("username", kind="stable").reset_index(drop=True)
    result[FLOAT_COLUMNS] = result[FLOAT_COLUMNS].round(6)
    return result

## 5. Загрузка таблицы user_features.csv

In [13]:
user_features_df = pd.read_csv(USER_FEATURES_PATH)
print(f"user_features_df: {user_features_df.shape}")

user_features_df: (11, 27)


## 6. Построение таблицы user_activity_types.csv

In [14]:
user_activity_types_df = assign_listener_types(user_features_df)
print(f"user_activity_types_df: {user_activity_types_df.shape}")

user_activity_types_df: (11, 18)


## 7. Проверка структуры и сохранение результата

In [15]:
assert not user_activity_types_df.empty
assert user_activity_types_df["username"].is_unique
assert not user_activity_types_df["listener_type"].isna().any()
assert not user_activity_types_df["behavior_profile"].isna().any()

user_activity_types_df.to_csv(USER_ACTIVITY_TYPES_PATH, index=False)
print(f"Saved file: {USER_ACTIVITY_TYPES_PATH}")

Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_activity_types.csv


## 8. Предварительный просмотр итоговой таблицы

In [16]:
user_activity_types_df.head()

,username,listener_type,listener_type_description,behavior_profile,assignment_rule,behavior_rule,total_plays,active_days,active_days_share,avg_daily_plays,max_daily_plays,evening_share,night_share,weekend_share,main_day_part,peak_hour,session_count,avg_session_length
0,Babs_05,Интенсивный,Пользователь с высоким средним числом прослуши...,Смешанная активность,avg_daily_plays >= upper_quartile,share_gap < 0.10; top_share=evening_share; sec...,33695,31,1.000000,1086.935484,16302,0.397418,0.125182,0.696127,Вечер,18,203,100.039409
1,Knapster01,Интенсивный,Пользователь с высоким средним числом прослуши...,Смешанная активность,avg_daily_plays >= upper_quartile,share_gap < 0.10; top_share=evening_share; sec...,27015,30,0.967742,900.500000,11688,0.384379,0.126189,0.647603,Вечер,22,176,110.062500
2,Orlenay,Регулярный,Пользователь со стабильной активностью без экс...,Смешанная активность,default regular listener rule,share_gap < 0.10; top_share=evening_share; sec...,10123,31,1.000000,326.548387,3667,0.383977,0.099180,0.619085,Вечер,18,151,94.675497
3,eartle,Регулярный,Пользователь со стабильной активностью без экс...,Смешанная активность,default regular listener rule,share_gap < 0.10; top_share=day_share; second_...,20966,31,1.000000,676.322581,7878,0.365353,0.105409,0.583230,День,18,171,113.181287
4,franhale,Интенсивный,Пользователь с высоким средним числом прослуши...,Смешанная активность,avg_daily_plays >= upper_quartile,share_gap < 0.10; top_share=evening_share; sec...,32712,31,1.000000,1055.225806,15177,0.394167,0.117633,0.677336,Вечер,22,191,111.094241
